In [241]:
import pandas as pd
from tqdm import tqdm

In [242]:
def augustus_refs(df):
    out_df= pd.DataFrame()
    accessions=df["accession"].unique().tolist()
    for acc in accessions:    
        busco = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/busco_{acc}.txt", header=None)
        house = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/housekeeping_{acc}.txt", header=None)
        nlr = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/nlrs_{acc}.txt", header=None)

        busco["type"] = "busco"
        house["type"] = "housekeeping"
        nlr["type"] = "nlr"
        gene_type = pd.concat([busco,house,nlr])
        gene_type.columns = ["ref_isoform","type"]
        gene_type = gene_type.groupby("ref_isoform")["type"].apply(', '.join).reset_index()

        gene_type["ref_gene"] = gene_type["ref_isoform"].str.split(".").str[0]
        gene_type.loc[gene_type["type"] == "busco, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "housekeeping, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "busco, housekeeping"] = "housekeeping"

        sub=df.loc[df["accession"]==acc]
        sub = pd.merge(left=sub, right=gene_type, how='left', left_on='associated_gene', right_on='ref_gene')
        
        out_df=pd.concat([out_df,sub])
    return out_df

### Prepare the mapping of gene type to Sqanti gene

In [245]:
# full df
aug_ref=pd.read_pickle("pb_augustus_sqanti_full.pkl")
aug_map=aug_ref[["accession","gene","associated_gene","isoform","gene_read_support"]]
aug_map["association"] = aug_map["associated_gene"].str.split("_").str.len().copy()
# split by clear cases and different ones
clear = aug_map.loc[aug_map["association"] == 1]
unclear = aug_map.loc[aug_map["association"] != 1]
# remove duplicated TAMA genes (still keeps the rows with more than one reference gene association)
clear_genes=clear.drop(labels="isoform", axis=1).drop_duplicates()
# associate gene type
#IDs = augustus_refs(clear_genes)

/tmp/ipykernel_125067/602736232.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aug_map["association"] = aug_map["associated_gene"].str.split("_").str.len().copy()


### Real analysis (for accession)

In [246]:
# get associated genes
geneID=clear.loc[clear["accession"]=="at9900"]

In [247]:
geneID

,accession,gene,associated_gene,isoform,gene_read_support,association
0,at9900,PB.1,g2,PB.1.1,2,1
1,at9900,PB.1,g2,PB.1.2,2,1
2,at9900,PB.10,g12,PB.10.1,96,1
3,at9900,PB.10,g12,PB.10.10,96,1
4,at9900,PB.10,g12,PB.10.11,96,1
...,...,...,...,...,...,...
325924,at9900,PB.16856,g31817,PB.16856.8,35,1
325925,at9900,PB.16856,g31817,PB.16856.9,35,1
325926,at9900,PB.16857,g31818,PB.16857.1,7,1
325927,at9900,PB.16857,g31818,PB.16857.2,7,1


In [248]:
# get gene ID, present ore than once
x=geneID["gene"].value_counts() > 1
x=x.reset_index()
x.columns=["gene", "bool"]
dups=x.loc[x["bool"]==True,"gene"].tolist()

In [249]:
# get the genes from the clear dataframe
unclear_cans=geneID[geneID["gene"].isin(dups)]

In [250]:
unclear_cans

,accession,gene,associated_gene,isoform,gene_read_support,association
0,at9900,PB.1,g2,PB.1.1,2,1
1,at9900,PB.1,g2,PB.1.2,2,1
2,at9900,PB.10,g12,PB.10.1,96,1
3,at9900,PB.10,g12,PB.10.10,96,1
4,at9900,PB.10,g12,PB.10.11,96,1
...,...,...,...,...,...,...
325924,at9900,PB.16856,g31817,PB.16856.8,35,1
325925,at9900,PB.16856,g31817,PB.16856.9,35,1
325926,at9900,PB.16857,g31818,PB.16857.1,7,1
325927,at9900,PB.16857,g31818,PB.16857.2,7,1


In [ ]:
# add type to already unclear


In [210]:
## move them to the unclear list
unclear_acc = unclear.loc[unclear["accession"]=="at9900"]
unclear_fin = pd.concat([unclear_acc,unclear_cans])

,accession,gene,associated_gene,isoform,gene_read_support,association
227,at9900,PB.1008,novelGene_14,PB.1008.1,1,2
403,at9900,PB.1014,novelGene_15,PB.1014.7,23,2
507,at9900,PB.102,g132_g131,PB.102.24,96,2
518,at9900,PB.102,g132_g131,PB.102.34,96,2
737,at9900,PB.1034,novelGene_g1472_AS,PB.1034.1,1,3
...,...,...,...,...,...,...
325766,at9900,PB.16846,novelGene_1278,PB.16846.8,17,2
325767,at9900,PB.16846,novelGene_1279,PB.16846.9,17,2
325867,at9900,PB.16849,g31801_g31802,PB.16849.1,18,2
325898,at9900,PB.16853,g31812_g31811,PB.16853.1,4,2


In [162]:
geneID[geneID["gene"] == "PB.4434"]

,accession,gene,associated_gene,gene_read_support,association,ref_isoform,type,ref_gene
4290,at9900,PB.4434,g7783,1139,1,NaN,NaN,NaN
4291,at9900,PB.4434,g7781,1139,1,NaN,NaN,NaN
4292,at9900,PB.4434,g7786,1139,1,NaN,NaN,NaN
4293,at9900,PB.4434,g7789,1139,1,NaN,NaN,NaN
4294,at9900,PB.4434,g7791,1139,1,NaN,NaN,NaN


In [78]:
# load orf df
at9900=pd.read_pickle("sqanti/at9900_pb_augustus_sqanti_orf.pkl")

In [79]:
at9900

,Gene,n ORFs,n Isoforms,Sqanti ORFs,Isoforms,n_Isoforms_ORF,Subcategories,Read_Support,Read_Support_ORF
0,PB.1,0,0,[],[],[],[],[],[]
0,PB.10,16,33,[MERTNSIEMDRERLTAEMAFRDSSSAVIRIRRRLPDLLTSVKLKY...,"[[PB.10.1, PB.10.10, PB.10.12, PB.10.13, PB.10...","[12, 3, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[[mono-exon, mono-exon, mono-exon, mono-exon, ...","[[3, 5, 1, 19, 1, 22, 1, 16, 1, 1, 1, 4], [1, ...","[75, 3, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
0,PB.100,11,17,[MATKPGVLTDWPWTPLGSFKYIVIAPWAVHSTYRFVTDDPEKRDL...,"[[PB.100.1, PB.100.16, PB.100.3, PB.100.6, PB....","[5, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1]","[[alternative_3end5end, alternative_3end, alte...","[[1, 1, 10, 1, 3], [1, 1], [1, 1], [1], [1], [...","[16, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1]"
0,PB.1000,9,14,[MEGMLSSGDQQRLVSSFLEIAVGQTAETARQFLQATSWKLEEAIQ...,"[[PB.1000.1, PB.1000.2, PB.1000.3, PB.1000.4, ...","[5, 2, 1, 1, 1, 1, 1, 1, 1]","[[at_least_one_novel_splicesite, at_least_one_...","[[4, 12, 3, 2, 1], [1, 1], [1], [1], [1], [1],...","[22, 2, 1, 1, 1, 1, 1, 1, 1]"
0,PB.10000,1,1,[MLLDCWAIPCVLVLTWFYLKTKYRLMKISGVFICIVGVFMVVFSD...,[[PB.10000.1]],[1],[[intron_retention]],[[1]],[1]
...,...,...,...,...,...,...,...,...,...
0,PB.9995,1,5,[MEANKKAKPSYGDVISNLPNDLLCRILSYLSTKEAALTSILSKRW...,"[[PB.9995.1, PB.9995.2, PB.9995.3, PB.9995.4, ...",[5],"[[alternative_3end5end, alternative_3end5end, ...","[[1, 1, 1, 1, 1]]",[5]
0,PB.9996,5,5,[MAANVDNPQVSTLKLILVYLLIGLCSVLCLMVRSVCVVIMCMKSS...,"[[PB.9996.1], [PB.9996.2], [PB.9996.3], [PB.99...","[1, 1, 1, 1, 1]","[[at_least_one_novel_splicesite], [alternative...","[[1], [1], [1], [1], [1]]","[1, 1, 1, 1, 1]"
0,PB.9997,8,27,[MDDCSKDLINCLPDNLLCQILSNLSTKEAALTSLLSKRWRYLFAL...,"[[PB.9997.1, PB.9997.10, PB.9997.11, PB.9997.1...","[19, 2, 1, 1, 1, 1, 1, 1]","[[at_least_one_novel_splicesite, alternative_3...","[[1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1,...","[20, 2, 1, 1, 1, 1, 1, 1]"
0,PB.9998,9,19,[MKKAFGQTVRDLKRGVNKKVLKVPGIEQKVLDATSNESWGPHGSL...,"[[PB.9998.10, PB.9998.12, PB.9998.13, PB.9998....","[10, 2, 1, 1, 1, 1, 1, 1, 1]","[[at_least_one_novel_splicesite, at_least_one_...","[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1], [1], ...","[10, 2, 1, 1, 1, 1, 1, 1, 1]"


In [20]:
# merge the type to the df
at9900_type=pd.merge(left=at9900, right=geneID[["gene","accession","associated_gene","type"]], how='left', left_on='Gene', right_on='gene').drop(labels="gene", axis=1)

### Read support for each sqanti gene

In [22]:
nlr = at9900_type.loc[at9900_type["type"]=="nlr"]
busco = at9900_type.loc[at9900_type["type"]=="busco"]
house= at9900_type.loc[at9900_type["type"]=="housekeeping"]

In [24]:
nlr["Gene"].value_counts()

PB.11213    2
PB.10722    1
PB.2936     1
PB.3271     1
PB.3265     1
           ..
PB.14957    1
PB.14956    1
PB.14895    1
PB.14871    1
PB.9420     1
Name: Gene, Length: 123, dtype: int64

In [183]:
nlr["n ORFs"].mean()

8.838709677419354

In [184]:
busco["n ORFs"].mean()

8.545280960174578

In [185]:
house["n ORFs"].mean()

6.261006289308176

In [186]:
at9900_type["n ORFs"].mean()

8.551459024986896

In [188]:
nlr["n Isoforms"].mean()

15.274193548387096

In [189]:
busco["n Isoforms"].mean()

18.222858701582105

In [190]:
house["n Isoforms"].mean()

13.515723270440251

In [191]:
at9900_type["n Isoforms"].mean()

19.29599860213175

In [211]:
for i in range(len(at9900_type)):
    if len(at9900_type.iloc[i]["Isoforms"]) == 0:
        continue
    else:
        for orf in at9900_type.iloc[i]["Isoforms"]:
            print(orf)
    #print(len(at9900_type.iloc[i]["Isoforms"]))

hi
['PB.10.1', 'PB.10.10', 'PB.10.12', 'PB.10.13', 'PB.10.14', 'PB.10.21', 'PB.10.25', 'PB.10.28', 'PB.10.3', 'PB.10.32', 'PB.10.33', 'PB.10.5']
['PB.10.11', 'PB.10.22', 'PB.10.30']
['PB.10.16', 'PB.10.19']
['PB.10.17', 'PB.10.24']
['PB.10.18', 'PB.10.31']
['PB.10.26', 'PB.10.8']
['PB.10.15']
['PB.10.2']
['PB.10.20']
['PB.10.23']
['PB.10.27']
['PB.10.29']
['PB.10.4']
['PB.10.6']
['PB.10.7']
['PB.10.9']
['PB.100.1', 'PB.100.16', 'PB.100.3', 'PB.100.6', 'PB.100.7']
['PB.100.10', 'PB.100.14']
['PB.100.11', 'PB.100.2']
['PB.100.12']
['PB.100.13']
['PB.100.15']
['PB.100.17']
['PB.100.4']
['PB.100.5']
['PB.100.8']
['PB.100.9']
['PB.1000.1', 'PB.1000.2', 'PB.1000.3', 'PB.1000.4', 'PB.1000.6']
['PB.1000.5', 'PB.1000.8']
['PB.1000.10']
['PB.1000.11']
['PB.1000.12']
['PB.1000.13']
['PB.1000.14']
['PB.1000.7']
['PB.1000.9']
['PB.10000.1']
['PB.10001.1', 'PB.10001.10', 'PB.10001.11', 'PB.10001.12', 'PB.10001.17', 'PB.10001.18', 'PB.10001.2', 'PB.10001.26', 'PB.10001.5', 'PB.10001.7', 'PB.10001.9']

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



# TAMA

In [97]:
# full df
aug_ref=pd.read_pickle("tama_augustus_sqanti_full.pkl")
aug_map=aug_ref[["accession","gene","associated_gene","isoform"]]

out_df=pd.DataFrame()
for acc in aug_map["accession"].unique().tolist():
    acc_df=aug_map.loc[aug_map["accession"] == acc]
    acc_df["association"] = acc_df["associated_gene"].str.split("_").str.len().copy()
    out_df=pd.concat([out_df, acc_df])
    
# split by clear cases and different ones
clear = out_df.loc[out_df["association"] == 1]
unclear = out_df.loc[out_df["association"] != 1]

clear=clear.drop(labels="isoform", axis=1).drop_duplicates()

# associate gene type
IDs = augustus_refs(clear)

/tmp/ipykernel_125067/2085959111.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acc_df["association"] = acc_df["associated_gene"].str.split("_").str.len().copy()
/tmp/ipykernel_125067/2085959111.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acc_df["association"] = acc_df["associated_gene"].str.split("_").str.len().copy()
/tmp/ipykernel_125067/2085959111.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value

In [99]:
#relate refgenes and type
final=pd.DataFrame()
for i in ID["accession"].unique().tolist():
    acc_df=pd.DataFrame()
    acc_id=ID.loc[ID["accession"]==i]
    at=pd.read_pickle(f"sqanti/{i}_tama_augustus_sqanti_orf.pkl")
    at_type=pd.merge(left=at, right=acc_id[["gene","accession","associated_gene","type"]], how='left', left_on='Gene', right_on='gene').drop(labels="gene", axis=1)
    final=pd.concat([final,at_type])

In [103]:
final["read_support_sum"] = final["Read_Support_ORF"].apply(sum)

In [139]:
# Luisa helper
helper=pd.read_csv("luisa_helper_nlr.txt", header=None, names=["acc_gene"])
ID = pd.DataFrame()
ID["accession"] = helper["acc_gene"].str.split("_").str[0]
ID["gene"] = helper["acc_gene"].str.split("_").str[1]

In [140]:
ID["accession"].value_counts().sort_index()

at6137    6
at6923    7
at6929    7
at7143    6
at8285    7
at9104    7
at9336    7
at9503    7
at9578    6
at9744    7
at9762    5
at9806    7
at9830    6
at9847    7
at9852    7
at9879    6
at9883    7
at9900    7
Name: accession, dtype: int64

In [141]:
ID["accession"].unique().tolist()[1:]

['at6923',
 'at6929',
 'at7143',
 'at8285',
 'at9104',
 'at9336',
 'at9503',
 'at9578',
 'at9744',
 'at9762',
 'at9806',
 'at9830',
 'at9847',
 'at9852',
 'at9879',
 'at9883',
 'at9900']

In [142]:
final

,Gene,n ORFs,n Isoforms,Sqanti ORFs,Isoforms,n_Isoforms_ORF,Subcategories,Read_Support,Read_Support_ORF,accession,associated_gene,type,read_support_sum
0,G1,1,2,[MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEVNICSY...,"[[G1.1, G1.2]]",[2],"[[alternative_3end5end, alternative_3end5end]]","[[1, 1]]",[2],at6923,g1,NaN,2
1,G10,5,22,[MATAFAPTKLTATVPLHGSQENRLLLPIRLAPPSSFLGSTRSLSL...,"[[G10.1, G10.11, G10.13, G10.17, G10.18, G10.1...","[15, 3, 2, 1, 1]","[[alternative_3end5end, alternative_3end, alte...","[[3, 2, 54, 3, 1, 17, 41, 11, 1, 1, 18, 1, 1, ...","[175, 3, 2, 1, 1]",at6923,g9,NaN,182
2,G100,1,3,[MNKNQTVGLDSEDDDSDDYDIAQVNCELALVEGQLCNIPYELYDL...,"[[G100.1, G100.2, G100.3]]",[3],"[[multi-exon, multi-exon, multi-exon]]","[[2, 1, 1]]",[4],at6923,g131,NaN,4
3,G1000,4,12,[MVSQIISCLSQSSSLLCLSDSRRLILPKTCTCGGLSRFSPGFASR...,"[[G1000.1, G1000.11, G1000.2, G1000.3, G1000.6...","[7, 2, 2, 1]","[[alternative_3end, alternative_3end, alternat...","[[1, 8, 1, 1, 1, 2, 1], [1, 1], [1, 1], [1]]","[15, 2, 2, 1]",at6923,g1385,NaN,20
4,G10000,2,8,[MDTVNDSFSPGSSRPSPATLSREDCWSEEATFTLIQAWGNRYVDL...,"[[G10000.1, G10000.3, G10000.4, G10000.5, G100...","[5, 3]","[[multi-exon, multi-exon, multi-exon, multi-ex...","[[1, 1, 1, 3, 1], [1, 1, 1]]","[7, 3]",at6923,g19407,NaN,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16622,G9995,8,39,[MNFGSLFDNTPGGGSTGARLLSGLSYGNHTAATNVLPGGAMAQAA...,"[[G9995.1, G9995.11, G9995.12, G9995.13, G9995...","[20, 5, 4, 3, 3, 2, 1, 1]","[[alternative_3end, alternative_5end, referenc...","[[1, 2, 1, 2, 1, 3, 5, 1, 3, 2, 7, 1, 1, 1, 1,...","[41, 5, 4, 3, 3, 2, 1, 1]",col0,AT4G00730,NaN,60
16623,G9996,4,34,[MGHVNLPASKRGNPRQWRLLDIVTAAFFGIVLLFFILLFTPLGDS...,"[[G9996.1, G9996.10, G9996.11, G9996.12, G9996...","[31, 1, 1, 1]","[[at_least_one_novel_splicesite, alternative_5...","[[1, 1, 1, 1, 1, 1, 6, 2, 1, 1, 2, 1, 1, 1, 1,...","[57, 1, 1, 1]",col0,AT4G00740,NaN,60
16624,G9997,3,6,[MGNYRWPSKLSKLSLRAKQTNLYRVILIAILCVTFYFVGVWQHSG...,"[[G9997.1, G9997.3, G9997.4, G9997.5], [G9997....","[4, 1, 1]","[[reference_match, alternative_3end, reference...","[[3, 1, 1, 1], [1], [1]]","[6, 1, 1]",col0,AT4G00750,NaN,8
16625,G9998,2,6,[MVSPTRDAIQSYMSITGASESLAIQRLEEHGNNLPEAINAHFRDV...,"[[G9998.1, G9998.2, G9998.3, G9998.4, G9998.5]...","[5, 1]","[[reference_match, reference_match, alternativ...","[[1, 1, 1, 1, 3], [1]]","[7, 1]",col0,AT4G00752,busco,8


In [143]:
out=pd.DataFrame()
for i in ID["accession"].unique().tolist():
    acc_df=pd.DataFrame()
    acc_gene=ID.loc[ID["accession"]==i]
    acc_final=final.loc[final["accession"]==i]
    at_nlr=acc_final[acc_final["associated_gene"].isin(acc_gene["gene"])]
    out=pd.concat([out,at_nlr])

In [144]:
out["accession"].value_counts().sort_index()

at6923    7
at6929    6
at7143    6
at8285    6
at9104    7
at9336    5
at9503    6
at9578    6
at9744    7
at9762    5
at9806    7
at9830    5
at9847    6
at9852    7
at9879    6
at9883    7
at9900    7
Name: accession, dtype: int64

In [145]:
x=out[["accession", "Gene","associated_gene","n Isoforms", "read_support_sum"]]

In [146]:
x

,accession,Gene,associated_gene,n Isoforms,read_support_sum
2724,at6923,G12377,g24288,28,107
3618,at6923,G13159,g25379,36,108
7677,at6923,G16726,g32065,17,18
7698,at6923,G16745,g32097,23,29
7699,at6923,G16746,g32098,22,27
...,...,...,...,...,...
7516,at9900,G16648,g31656,10,10
7540,at9900,G16669,g31688,18,19
7543,at9900,G16671,g31689,13,15
8181,at9900,G2157,g3277,2,3


In [147]:
x.to_csv("helper_nlr_read_support.csv", index=False)